# Data Preprocessing & Feature Engineering

In [1]:
import os
import pandas as pd
import numpy as np
import holidays

In [2]:
# Pathing to raw data
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir) if current_dir.endswith('notebooks') else current_dir
data_path = os.path.join(project_root, 'data', 'raw', 'synthetic_mobility_data.csv')

# Load and parse
df = pd.read_csv(data_path)
date_format = '%b %d %Y %I:%M%p'
df['PickupDateTimeScheduled'] = pd.to_datetime(df['PickupDateTimeScheduled'], format=date_format)

# Extract clean operational Date (time-series anchor)
df['Date'] = df['PickupDateTimeScheduled'].dt.date

# Aggregate: Daily Volume per Vehicle Type
daily_demand = df.groupby(['Date', 'PrefferedVehicleType']).size().reset_index(name='TotalTrips')

# Ensure Date is a proper datetime object for sorting and splitting
daily_demand['Date'] = pd.to_datetime(daily_demand['Date'])
daily_demand = daily_demand.sort_values(by=['Date', 'PrefferedVehicleType']).reset_index(drop=True)

print(f"Prototyping Dataset Shape: {daily_demand.shape}")
display(daily_demand.head(10))

Prototyping Dataset Shape: (3356, 3)


,Date,PrefferedVehicleType,TotalTrips
0,2024-01-01,MID-MINI,1
1,2024-01-01,MINI,3
2,2024-01-01,SEDN,17
3,2024-01-01,SUV,11
4,2024-01-01,Sprinter,2
5,2024-01-02,MC,1
6,2024-01-02,MID-MINI,1
7,2024-01-02,SEDN,5
8,2024-01-02,SUV,5
9,2024-01-03,MC,1


In [3]:
# Sort by Vehicle Type and Date first
daily_demand = daily_demand.sort_values(by=['PrefferedVehicleType', 'Date']).reset_index(drop=True)

# Lag Features (Historical Memory)
daily_demand['Demand_7_Days_Ago'] = daily_demand.groupby('PrefferedVehicleType')['TotalTrips'].shift(7)
daily_demand['Demand_14_Days_Ago'] = daily_demand.groupby('PrefferedVehicleType')['TotalTrips'].shift(14)

# Drop the rows with missing history (the first 14 days)
model_df = daily_demand.dropna().reset_index(drop=True)

# 4. Time-Series Feature Engineering
model_df['DayOfWeek'] = model_df['Date'].dt.dayofweek
model_df['Month'] = model_df['Date'].dt.month
model_df['Year'] = model_df['Date'].dt.year

# Categorical Feature Engineering (One-Hot Encoding)
model_df = pd.get_dummies(
    model_df, 
    columns=['PrefferedVehicleType'], 
    prefix='type', 
    dtype=int
)

# We leave the 'Date' column intact for the split
print(f"Engineered Dataset Shape: {model_df.shape}")
display(model_df.head())

Engineered Dataset Shape: (3272, 13)


,Date,TotalTrips,Demand_7_Days_Ago,Demand_14_Days_Ago,DayOfWeek,Month,Year,type_MC,type_MID-MINI,type_MINI,type_SEDN,type_SUV,type_Sprinter
0,2024-03-15,1,1.0,1.0,4,3,2024,1,0,0,0,0,0
1,2024-03-18,1,1.0,1.0,0,3,2024,1,0,0,0,0,0
2,2024-03-19,1,3.0,1.0,1,3,2024,1,0,0,0,0,0
3,2024-03-20,1,1.0,1.0,2,3,2024,1,0,0,0,0,0
4,2024-03-23,1,1.0,1.0,5,3,2024,1,0,0,0,0,0


In [4]:
# Initialize the US holiday calendar
us_holidays = holidays.US()

# Map the dates to a binary 1/0 flag
# If the Date is in the holiday calendar, return 1, else 0
model_df['is_Holiday'] = model_df['Date'].apply(lambda x: 1 if x in us_holidays else 0)

print(f"Holiday feature added.. Found {model_df['is_Holiday'].sum()} holiday occurrences in the dataset.")

# Display new column appended
display(model_df[['Date', 'TotalTrips', 'is_Holiday']].head(10))

Holiday feature added.. Found 104 holiday occurrences in the dataset.


,Date,TotalTrips,is_Holiday
0,2024-03-15,1,0
1,2024-03-18,1,0
2,2024-03-19,1,0
3,2024-03-20,1,0
4,2024-03-23,1,0
5,2024-03-25,1,0
6,2024-04-01,2,0
7,2024-04-03,1,0
8,2024-04-05,1,0
9,2024-04-08,3,0


In [7]:
# Sanity check 
display(model_df.head())

,Date,TotalTrips,Demand_7_Days_Ago,Demand_14_Days_Ago,DayOfWeek,Month,Year,type_MC,type_MID-MINI,type_MINI,type_SEDN,type_SUV,type_Sprinter,is_Holiday
0,2024-03-15,1,1.0,1.0,4,3,2024,1,0,0,0,0,0,0
1,2024-03-18,1,1.0,1.0,0,3,2024,1,0,0,0,0,0,0
2,2024-03-19,1,3.0,1.0,1,3,2024,1,0,0,0,0,0,0
3,2024-03-20,1,1.0,1.0,2,3,2024,1,0,0,0,0,0,0
4,2024-03-23,1,1.0,1.0,5,3,2024,1,0,0,0,0,0,0


# Dataset Train Test Split

In [9]:
# Define the split threshold as a simple string
split_date = '2026-01-01'

# Slice the data chronologically
train_df = model_df[model_df['Date'] < split_date].copy()
test_df = model_df[model_df['Date'] >= split_date].copy()

# Isolate Features (X) from the Target Variable (y)
X_train = train_df.drop(columns=['TotalTrips', 'Date'])
y_train = train_df['TotalTrips']

X_test = test_df.drop(columns=['TotalTrips', 'Date'])
y_test = test_df['TotalTrips']

print(f"Training Data: {X_train.shape[0]} rows (Date range: {train_df['Date'].min().date()} to {train_df['Date'].max().date()})")
print(f"Testing Data:  {X_test.shape[0]} rows (Date range: {test_df['Date'].min().date()} to {test_df['Date'].max().date()})")

# Verify the final feature matrix
print("\nFeatures ready for modeling:")
print(X_train.columns.tolist())

Training Data: 2769 rows (Date range: 2024-01-15 to 2025-12-31)
Testing Data:  503 rows (Date range: 2026-01-01 to 2026-04-30)

Features ready for modeling:
['Demand_7_Days_Ago', 'Demand_14_Days_Ago', 'DayOfWeek', 'Month', 'Year', 'type_MC', 'type_MID-MINI', 'type_MINI', 'type_SEDN', 'type_SUV', 'type_Sprinter', 'is_Holiday']


# Linear Regression (Baseline)

In [11]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [12]:
# Set up local MLflow Experiment
mlflow.set_experiment("Mobility_Demand_Forecasting")

# Start the tracking run
with mlflow.start_run(run_name="Baseline_Linear_Regression"):
    
    # Initialize and Train the Model
    lr_model = LinearRegression()
    lr_model.fit(X_train, y_train)
    
    # Make Predictions on the unseen 2026 Test Data
    y_pred = lr_model.predict(X_test)
    
    # Calculate Error Metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    # Log parameters and metrics to MLflow
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("features_count", X_train.shape[1])
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    
    # Save the trained model artifact
    mlflow.sklearn.log_model(lr_model, "linear_regression_baseline")
    
    print("Baseline Model Trained & Tracked in MLflow..")
    print("-" * 40)
    print(f"Mean Absolute Error (MAE): {mae:.2f} trips")
    print(f"Root Mean Squared Error (RMSE): {rmse:.2f} trips")

2026/05/15 12:50:46 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/15 12:50:46 INFO mlflow.store.db.utils: Updating database tables
2026/05/15 12:50:50 INFO mlflow.tracking.fluent: Experiment with name 'Mobility_Demand_Forecasting' does not exist. Creating a new experiment.
2026/05/15 12:50:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/15 12:50:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Baseline Model Trained & Tracked in MLflow..
----------------------------------------
Mean Absolute Error (MAE): 2.78 trips
Root Mean Squared Error (RMSE): 4.36 trips


# XGBoost Model Comparison

In [13]:
import xgboost as xgb
import mlflow.xgboost

In [14]:
# xgb new run in the same experiment
with mlflow.start_run(run_name="XGBoost_Baseline"):
    
    # Initialize XGBoost Regressor / default hyperparameters
    xgb_model = xgb.XGBRegressor(
        n_estimators=100,        # Number of trees to build
        learning_rate=0.1,       # How each tree corrects the last one
        max_depth=5,             # Maximum depth of each decision tree
        random_state=42,
        objective='reg:squarederror'
    )
    
    # Train Model
    xgb_model.fit(X_train, y_train)
    
    # Make Predictions on Test Data
    y_pred_xgb = xgb_model.predict(X_test)
    
    # Calculate Metrics
    mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
    rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
    
    # Log parameters and metrics to MLflow
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("max_depth", 5)
    
    mlflow.log_metric("MAE", mae_xgb)
    mlflow.log_metric("RMSE", rmse_xgb)
    
    # Save the XGBoost model artifact
    mlflow.xgboost.log_model(xgb_model, "xgboost_model")
    
    # Print results
    print("XGBoost Model Trained & Tracked..")
    print("-" * 50)
    print(f"Linear Regression RMSE (The Floor): 4.36 trips")
    print(f"XGBoost RMSE (The Challenger):    {rmse_xgb:.2f} trips")
    print("-" * 50)
    
    if rmse_xgb < 4.36:
        improvement = ((4.36 - rmse_xgb) / 4.36) * 100
        print(f"Success: XGBoost improved accuracy by {improvement:.1f}%..")
    else:
        print("XGBoost did not beat baseline. Need to check features.")

2026/05/15 15:52:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


XGBoost Model Trained & Tracked..
--------------------------------------------------
Linear Regression RMSE (The Floor): 4.36 trips
XGBoost RMSE (The Challenger):    4.43 trips
--------------------------------------------------
XGBoost did not beat baseline. Need to check features.


# XGBoost Model Tuning

In [15]:
# New run to track tuning attempt(s)
with mlflow.start_run(run_name="XGBoost_Tuned_v1"):
    
    # Initialize a strictly Regularized XGBoost
    xgb_model_tuned = xgb.XGBRegressor(
        n_estimators=150,        # Slightly more trees..
        learning_rate=0.05,      # Slower learning
        max_depth=3,             # Shallower trees (prevent overfitting)
        min_child_weight=5,      # Requires more evidence before making a split
        subsample=0.8,           # Randomly drops 20% of rows per tree
        colsample_bytree=0.8,    # Randomly drops 20% of columns per tree
        random_state=42,
        objective='reg:squarederror'
    )
    
    # Train Tuned Model
    xgb_model_tuned.fit(X_train, y_train)
    
    # Make Predictions on Test Data
    y_pred_tuned = xgb_model_tuned.predict(X_test)
    
    # Calculate Metrics
    mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
    rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
    
    # Log parameters and metrics
    mlflow.log_param("model_type", "XGBoost_Tuned")
    mlflow.log_param("max_depth", 3)
    mlflow.log_param("learning_rate", 0.05)
    
    mlflow.log_metric("MAE", mae_tuned)
    mlflow.log_metric("RMSE", rmse_tuned)
    
    # Save the artifact
    mlflow.xgboost.log_model(xgb_model_tuned, "xgboost_model_tuned")
    
    # Print the Showdown Results
    print("Tuned XGBoost Model Trained & Tracked..")
    print("-" * 50)
    print(f"Linear Regression RMSE (The Floor): 4.36 trips")
    print(f"XGBoost Baseline RMSE:            4.43 trips")
    print(f"XGBoost Tuned RMSE:               {rmse_tuned:.2f} trips")
    print("-" * 50)
    
    if rmse_tuned < 4.36:
        improvement = ((4.36 - rmse_tuned) / 4.36) * 100
        print(f"Success: Beat the baseline. Improved accuracy by {improvement:.1f}%..")
    else:
        print("Still trailing baseline..")

2026/05/15 15:58:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Tuned XGBoost Model Trained & Tracked..
--------------------------------------------------
Linear Regression RMSE (The Floor): 4.36 trips
XGBoost Baseline RMSE:            4.43 trips
XGBoost Tuned RMSE:               4.36 trips
--------------------------------------------------
Still trailing baseline..


# XGBoost Model Optimization (More Tuning)

In [18]:
# New final tracking run
with mlflow.start_run(run_name="XGBoost_Poisson_Optimized"):
    
    # Initialize Poisson-Optimized XGBoost
    xgb_poisson = xgb.XGBRegressor(
        n_estimators=150,
        learning_rate=0.08,      # Slightly faster learning now that it has the right math
        max_depth=4,             # Giving a bit more room to map the spikes
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        objective='count:poisson' # Hekp capture volume spikes/surge
    )
    
    # Train Model
    xgb_poisson.fit(X_train, y_train)
    
    # Make Predictions
    y_pred_poisson = xgb_poisson.predict(X_test)
    
    # Calculate Metrics
    mae_poisson = mean_absolute_error(y_test, y_pred_poisson)
    rmse_poisson = np.sqrt(mean_squared_error(y_test, y_pred_poisson))
    
    # Log to MLflow
    mlflow.log_param("model_type", "XGBoost_Poisson")
    mlflow.log_param("objective", "count:poisson")
    mlflow.log_metric("MAE", mae_poisson)
    mlflow.log_metric("RMSE", rmse_poisson)
    
    # Save winning artifact
    mlflow.xgboost.log_model(xgb_poisson, "xgboost_poisson_model")
    
    # Final attempt
    print("Poisson-Optimized XGBoost Model Trained..")
    print("-" * 50)
    print(f"Linear Regression RMSE (Baseline): 4.36 trips")
    print(f"XGBoost Poisson RMSE:             {rmse_poisson:.2f} trips")
    print("-" * 50)
    
    if rmse_poisson < 4.36:
        improvement = ((4.36 - rmse_poisson) / 4.36) * 100
        print(f"Success: Baseline improved by {improvement:.1f}%..")
        print("Model understands true shape corporate mobility demand..")
    else:
        print("Still trailing baseline..")

2026/05/15 16:11:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Poisson-Optimized XGBoost Model Trained..
--------------------------------------------------
Linear Regression RMSE (Baseline): 4.36 trips
XGBoost Poisson RMSE:             4.38 trips
--------------------------------------------------
Still trailing baseline..
